# NB_bishop_ch17_figures
## Key Figures for Bishop Chapter 17 -- Generative Adversarial Networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_17/NB_bishop_ch17_figures.ipynb)

Generate publication-quality figures for the GAN framework, training dynamics, and distribution evolution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

COLORS = {'blue': '#1A3A6E', 'red': '#CD0000', 'green': '#2E7D32', 'amber': '#B5853F'}

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            legend._ncols = min(len(legend.get_texts()), 4)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

np.random.seed(42)

## Figure 17.1 -- GAN Framework Diagram

z -> Generator -> fake_x, real_x -> Discriminator -> real/fake.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(-1, 16)
ax.set_ylim(-1, 7)
ax.axis('off')

def draw_box(ax, x, y, w, h, label, fc, ec, fs=11):
    box = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                                   facecolor=fc, edgecolor=ec, linewidth=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=fs, fontweight='bold', color=ec)

# Latent z
draw_box(ax, 0, 2.5, 2, 1.2, 'Latent\n$\mathbf{z} \sim p(z)$', '#E8E8E8', '#333333', 12)

# Generator
draw_box(ax, 3.5, 2.5, 2.5, 1.2, 'Generator\n$G(z)$', '#D6E4F0', COLORS['blue'], 12)

# Arrow z -> G
ax.annotate('', xy=(3.5, 3.1), xytext=(2.0, 3.1),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Fake x
draw_box(ax, 7.5, 2.5, 2, 1.2, 'Fake $\hat{x}$', '#FCE4EC', COLORS['red'], 12)

# Arrow G -> fake
ax.annotate('', xy=(7.5, 3.1), xytext=(6.0, 3.1),
            arrowprops=dict(arrowstyle='->', color=COLORS['blue'], lw=2))

# Real x
draw_box(ax, 7.5, 5.0, 2, 1.2, 'Real $x$', '#E8F5E9', COLORS['green'], 12)

# Discriminator
draw_box(ax, 11, 3.5, 2.5, 1.5, 'Discriminator\n$D(x)$', '#FFF3E0', COLORS['amber'], 12)

# Arrow fake -> D
ax.annotate('', xy=(11, 4.0), xytext=(9.5, 3.1),
            arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=2))

# Arrow real -> D
ax.annotate('', xy=(11, 4.5), xytext=(9.5, 5.6),
            arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=2))

# Output
draw_box(ax, 14.5, 3.5, 1.2, 1.5, 'Real\n/\nFake', '#E8E8E8', '#333333', 11)

# Arrow D -> output
ax.annotate('', xy=(14.5, 4.25), xytext=(13.5, 4.25),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Feedback arrows
ax.annotate('', xy=(4.75, 2.5), xytext=(12.25, 3.5),
            arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=1.5,
                            linestyle='--', connectionstyle='arc3,rad=0.4'))
ax.text(8.5, 1.2, 'Adversarial feedback', fontsize=10, ha='center',
        color=COLORS['red'], fontstyle='italic')

ax.set_title('Generative Adversarial Network Framework', fontweight='bold', fontsize=15, pad=15)

fig.tight_layout()
save_fig(fig, 'fig17_1_gan_framework')
plt.show()

## Figure 17 -- GAN Training Curves

Generator and discriminator loss over training epochs.

In [ ]:
np.random.seed(42)
epochs = np.arange(200)

# Simulate realistic GAN training losses
# Discriminator: starts low (easy to classify), rises as G improves, settles near log(2)
d_loss = 0.1 + 0.5 * (1 - np.exp(-epochs / 40)) + 0.05 * np.random.randn(200)
d_loss = np.clip(d_loss, 0.05, None)

# Generator: starts high (poor fakes), decreases as G learns
g_loss = 3.0 * np.exp(-epochs / 50) + 0.7 + 0.08 * np.random.randn(200)
g_loss = np.clip(g_loss, 0.3, None)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, d_loss, color=COLORS['blue'], linewidth=2, label='Discriminator Loss', alpha=0.9)
ax.plot(epochs, g_loss, color=COLORS['red'], linewidth=2, label='Generator Loss', alpha=0.9)
ax.axhline(y=np.log(2), color=COLORS['amber'], linestyle='--', linewidth=1.5,
           label=f'Nash Equilibrium $\\ln 2 \\approx {np.log(2):.2f}$')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('GAN Training Dynamics', fontweight='bold', fontsize=14)
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig17_gan_training')
plt.show()

## Figure 17 -- GAN Distribution Evolution

Real data distribution, generator distribution, and discriminator boundary evolving over training (early, mid, converged).

In [ ]:
x_range = np.linspace(-4, 6, 500)

# Real distribution: mixture of two Gaussians
p_real = 0.5 * np.exp(-0.5 * ((x_range - 1) / 0.6)**2) / (0.6 * np.sqrt(2*np.pi)) + \
         0.5 * np.exp(-0.5 * ((x_range - 3) / 0.5)**2) / (0.5 * np.sqrt(2*np.pi))

# Generator distributions at different stages
# Early: broad, poorly matched
p_gen_early = np.exp(-0.5 * ((x_range - 0) / 1.5)**2) / (1.5 * np.sqrt(2*np.pi))
# Mid: getting closer but mode collapse tendency
p_gen_mid = 0.7 * np.exp(-0.5 * ((x_range - 1.5) / 0.8)**2) / (0.8 * np.sqrt(2*np.pi)) + \
            0.3 * np.exp(-0.5 * ((x_range - 3.0) / 0.9)**2) / (0.9 * np.sqrt(2*np.pi))
# Converged: closely matches real
p_gen_conv = 0.48 * np.exp(-0.5 * ((x_range - 1.05) / 0.62)**2) / (0.62 * np.sqrt(2*np.pi)) + \
             0.52 * np.exp(-0.5 * ((x_range - 2.95) / 0.52)**2) / (0.52 * np.sqrt(2*np.pi))

# Discriminator outputs
d_early = 1.0 / (1 + np.exp(-(2 * (p_real - p_gen_early) / (p_real + p_gen_early + 1e-8))))
d_mid = 1.0 / (1 + np.exp(-(0.8 * (p_real - p_gen_mid) / (p_real + p_gen_mid + 1e-8))))
d_conv = np.ones_like(x_range) * 0.5  # Converged: D = 0.5 everywhere

stages = [
    ('Early Training', p_gen_early, d_early),
    ('Mid Training', p_gen_mid, d_mid),
    ('Converged', p_gen_conv, d_conv),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=False)

for ax, (title, p_gen, d_out) in zip(axes, stages):
    ax.fill_between(x_range, p_real, alpha=0.15, color=COLORS['blue'])
    ax.plot(x_range, p_real, color=COLORS['blue'], linewidth=2, label='$p_{data}(x)$')
    ax.fill_between(x_range, p_gen, alpha=0.15, color=COLORS['red'])
    ax.plot(x_range, p_gen, color=COLORS['red'], linewidth=2, linestyle='--', label='$p_G(x)$')

    ax2 = ax.twinx()
    ax2.plot(x_range, d_out, color=COLORS['green'], linewidth=1.5, linestyle=':', label='$D(x)$')
    ax2.set_ylim(0, 1.05)
    ax2.set_ylabel('$D(x)$', color=COLORS['green'], fontsize=10)
    ax2.tick_params(axis='y', colors=COLORS['green'])
    ax2.grid(False)

    ax.set_xlabel('$x$', fontsize=11)
    ax.set_title(title, fontweight='bold', fontsize=12)
    if ax == axes[0]:
        ax.set_ylabel('Density', fontsize=11)

# Shared legend
handles1 = [plt.Line2D([0], [0], color=COLORS['blue'], lw=2, label='$p_{data}(x)$'),
            plt.Line2D([0], [0], color=COLORS['red'], lw=2, ls='--', label='$p_G(x)$'),
            plt.Line2D([0], [0], color=COLORS['green'], lw=1.5, ls=':', label='$D(x)$')]
fig.legend(handles=handles1, loc='upper center', bbox_to_anchor=(0.5, -0.02),
           ncol=3, framealpha=0.0, fontsize=11)

fig.suptitle('GAN Training: Distribution Evolution', fontweight='bold', fontsize=14, y=1.03)
fig.tight_layout()
save_fig(fig, 'fig17_gan_distributions')
plt.show()

## Summary

- **fig17_1**: GAN architecture diagram showing generator and discriminator data flow
- **fig17_gan_training**: Training loss curves with Nash equilibrium reference
- **fig17_gan_distributions**: Distribution evolution from early to converged training